In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
import numba as nb

totalNum = 369
globalNum, xCoordNano, yCoordNano, boxPosition = np.loadtxt("data/11.dat", unpack = True)
# Box position mapping: 0 - inside, 1 - left/right edge, 2 - bottom edge, 3 - top edge

cellNum, localNum, globalNum2 = np.loadtxt("data/19.dat", unpack = True)

def get_p(index):
  match index:
    case 1:
      p = - np.sqrt(3/7 - 2/7 * np.sqrt(6/5))
    case 2:
      p = np.sqrt(3/7 - 2/7 * np.sqrt(6/5))
    case 3:
      p = np.sqrt(3/7 + 2/7 * np.sqrt(6/5))
    case 4:
      p = - np.sqrt(3/7 + 2/7 * np.sqrt(6/5))
  return p

def get_w(index):
  match index:
    case 1 | 2:
      w = (18 + np.sqrt(30)) / 36
    case 3 | 4:
      w = (18 - np.sqrt(30)) / 36
  return w

def get_global_coord_from_local(epsilon1, epsilon2, k):
  xGlobal, yGlobal = 0, 0
  for i in range(1, 4 + 1):
    xGlobal += get_g_shape_function(i, epsilon1, epsilon2) * map_k_and_i_to_global_coord(k, i, "x")
    yGlobal += get_g_shape_function(i, epsilon1, epsilon2) * map_k_and_i_to_global_coord(k, i, "y")
  return xGlobal, yGlobal

# def get_d_nlg(k, i):
#   return d

# def get_global_wave_function_from_local(epsilon1, epsilon2, k):
#   returnValue = 0
#   for i in range(1, 9 + 1):
#     returnValue += get_h_shape_function(i, epsilon1, epsilon2) * get_d_nlg(k, i)
#   return returnValue

def map_k_and_i_to_global_coord(k, i, coord):
  searchedglobalNum = np.where( (cellNum == k) & (localNum == i) )[0][0]
  match coord:
    case "x":
      global_coord = xCoordNano[int(globalNum2[searchedglobalNum]) - 1]
    case "y":
      global_coord = yCoordNano[int(globalNum2[searchedglobalNum]) - 1]
  return global_coord


def get_g_shape_function(index, epsilon1, epsilon2):
  f1 = lambda e: .5 * (1 - e)
  f2 = lambda e: .5 * (1 + e)
  match index:
    case 1:
      shapeFunction = f1(epsilon1) * f1(epsilon2)
    case 2:
      shapeFunction = f2(epsilon1) * f1(epsilon2)
    case 3:
      shapeFunction = f1(epsilon1) * f2(epsilon2)
    case 4:
      shapeFunction = f2(epsilon1) * f2(epsilon2)
  return shapeFunction

def get_h_shape_function(index, epsilon1, epsilon2):
  q1 = lambda e: .5 * e * (e - 1)
  q2 = lambda e: (1 - e) * (1 + e)
  q3 = lambda e: .5 * (1 + e)
  match index:
    case 1:
      shapeFunction = q1(epsilon1) * q1(epsilon2)
    case 2:
      shapeFunction = q3(epsilon1) * q1(epsilon2)
    case 3:
      shapeFunction = q1(epsilon1) * q3(epsilon2)
    case 4:
      shapeFunction = q3(epsilon1) * q3(epsilon2)
    case 5:
      shapeFunction = q2(epsilon1) * q1(epsilon2)
    case 6:
      shapeFunction = q3(epsilon1) * q2(epsilon2)
    case 7:
      shapeFunction = q1(epsilon1) * q2(epsilon2)
    case 8:
      shapeFunction = q2(epsilon1) * q3(epsilon2)
    case 9:
      shapeFunction = q2(epsilon1) * q2(epsilon2)
  return shapeFunction

def derive_h(whichCoord, whichIndex, coordX, coordY, Delta):
  match whichCoord:
    case "x":
      returnValue = .5 * (get_h_shape_function(whichIndex, coordX + Delta, coordY) - get_h_shape_function(whichIndex, coordX - Delta, coordY)) / Delta
    case "y":
      returnValue = .5 * (get_h_shape_function(whichIndex, coordX, coordY + Delta) - get_h_shape_function(whichIndex, coordX, coordY - Delta)) / Delta
  return returnValue

In [25]:
# TESTS
print(globalNum.shape[0] == xCoordNano.shape[0] == yCoordNano.shape[0] == boxPosition.shape[0] == 369)
print(cellNum.shape[0] == localNum.shape[0] == globalNum2.shape[0] == 720)
print(globalNum.dtype, xCoordNano.dtype, yCoordNano.dtype, boxPosition.dtype)
print(cellNum.dtype, localNum.dtype, globalNum2.dtype)
print(type(globalNum))
print(globalNum[0] == 1)

print((x1:= map_k_and_i_to_global_coord(80, 2, "x")) == 25.0)
print((y1:= map_k_and_i_to_global_coord(80, 2, "y")) == 37.5)
x2, y2 = get_global_coord_from_local(-0.5, -0.5, 80)
print(x2 == x1 - 2 * 6.25 * .75, y2 == y1 + 2 * 6.25 * .25)


True
True
float64 float64 float64 float64
float64 float64 float64
<class 'numpy.ndarray'>
True
True
True
True True
